In [1]:
import os
import pandas as pd
import numpy as np
import glob
import warnings

### ELABORAZIONE DEI DATI GREZZI E ESTRAZIONE DELLE METRICHE ###

In [2]:
percorso_sani_demo = r"controls - demographic+clinical - datasetv1.csv"
percorso_pd_demo   = r"pd - demographic+clinical - datasetv1.csv"
lista_df_clinici = []
for percorso, gruppo in [(percorso_sani_demo, 'Sano'), (percorso_pd_demo, 'Parkinson')]:
    df_temp = pd.read_csv(percorso, header=1, engine='python')
    df_temp.columns = df_temp.columns.str.strip()
    if gruppo == 'Sano':
        df_temp['Modified Hoehn & Yahr Score'] = 0
    else:
        df_temp['Modified Hoehn & Yahr Score'] = pd.to_numeric(
            df_temp['Modified Hoehn & Yahr Score'], errors='coerce'
        )
    if 'Age (years)' in df_temp.columns:
        df_temp = df_temp.rename(columns={'Age (years)': 'Age'})
    colonne_necessarie = ['Subject ID', 'Age', 'Height (in)', 'Weight (kg)', 'Sex','Modified Hoehn & Yahr Score']
    df_cols = df_temp[colonne_necessarie].copy()
    df_cols.columns = ['ID', 'Eta', 'Altezza', 'Peso', 'Sesso', 'H&Y']
    df_cols['Altezza'] = pd.to_numeric(df_cols['Altezza'], errors='coerce')
    df_cols['Altezza'] = (df_cols['Altezza'] * 2.54).round(1)    
    df_cols['Gruppo'] = gruppo
    df_cols['ID'] = df_cols['ID'].astype(str).str.upper().str.strip()    
    lista_df_clinici.append(df_cols)
df_clinico_totale = pd.concat(lista_df_clinici, ignore_index=True)
df_clinico_totale.to_csv("master_clinico_pulito.csv", index=False)

### codice per creare tre tabelle, una per test, con i dati utili ###

In [3]:
# DEFINIZIONE DELLE COSTANTI E DEI PATH DI SISTEMA

PATH_PD_CSV = r"PD PARTICIPANTS\CSV files"  # Directory dei pazienti patologici (Parkinson's Disease)
PATH_HC_CSV = r"CONTROL PARTICIPANTS\CSV files" # Directory dei soggetti di controllo sani (Healthy Controls)
FILE_MASTER_PULITO = r"master_clinico_pulito.csv" # Dataset contenente l'anagrafica clinica

SENSOR_SIZE_M = 0.0127 
# Risoluzione spaziale del tappeto sensorizzato (fattore di conversione in metri)
SAMPLING_RATE = 0.01 
# Frequenza di campionamento del sistema (100 Hz, ovvero un sample ogni 0.01 secondi)

def elabora_dati(file_path):
    
    # Acquisizione del dataset raw e normalizzazione degli header
    df = pd.read_csv(file_path, sep=None, engine='python')
    df.columns = df.columns.str.strip()
    
    # Filtraggio del segnale: isolamento del task motorio di interesse (Cammino/Walk)
    # Viene generato un subset contenente esclusivamente i frame temporali associati all'evento 'Walk'
    df_walk = df[df["GeneralEvent"].astype(str).str.contains('Walk')].copy()
    
    # Calcolo della durata complessiva del task motorio
    tempo = len(df_walk) * SAMPLING_RATE
    
    # Estrazione degli eventi di Heel Strike (Contatto Iniziale)
    # L'operatore di derivata discreta diff(1) rileva le transizioni di stato (da 0 a 1).
    # La condizione == 1 isola il frame esatto in cui il piede entra in contatto con il suolo, 
    # ignorando i frame successivi di stance (appoggio continuo).
    df_walk['L_strike'] = (df_walk["L Foot Contact"].diff(1) == 1)
    df_walk['R_strike'] = (df_walk["R Foot Contact"].diff(1) == 1)
    
    # Generazione di un dataframe ridotto contenente esclusivamente i frame di impatto plantare
    df_passi = df_walk[df_walk['L_strike'] | df_walk['R_strike']].copy()    
    
    def get_centroid(val):
        lista_pezzi = str(val).split('|')
        nums = []
        for x in lista_pezzi:
            numero = float(x)
            nums.append(numero)
        return np.mean(nums) * SENSOR_SIZE_M 

    # Applicazione del calcolo del centroide agli assi X (progressione) e Y (medio-laterale)
    cx = df_passi["Walkway_X"].apply(get_centroid).values
    cy = df_passi["Walkway_Y"].apply(get_centroid).values
    
    # Calcolo della distanza totale percorsa tramite sommatoria delle distanze euclidee
    # tra le coordinate dei centroidi in istanti di impatto successivi.
    distanza = np.sum(np.sqrt(np.diff(cx)**2 + np.diff(cy)**2))
    
    # Calcolo e restituzione delle metriche spazio-temporali del cammino
    return {
        'vel': distanza / tempo,                            # Velocità media (m/s)
        'fal': distanza / (len(df_passi) / 2),              # Lunghezza media della falcata / Stride length (m)
        'cad': (len(df_passi) / tempo) * 60,                # Cadenza (passi/minuto)
        'n': len(df_passi),                                 # Numero totale di passi
        'durata': tempo                                     # Durata del task (s)
    }


# MAIN PIPELINE: INTEGRAZIONE DATI CLINICI E CINEMATICI

# Caricamento del database anagrafico/clinico master
master = pd.read_csv(FILE_MASTER_PULITO)  
pazienti_list = []
    
# Iterazione sui due gruppi di studio: Patologici (PD) e Controlli (CO)
for cartella, gruppo_cartella in [(PATH_PD_CSV, 'PD'), (PATH_HC_CSV, 'CO')]:
    
    # Ricerca automatica di tutti i file di interesse (task motorio self-paced)
    files = glob.glob(os.path.join(cartella, "*_selfpace.csv")) 
    
    for f_path in files:
        # Estrazione dell'ID identificativo del soggetto tramite string parsing sul nome del file
        sid = os.path.basename(f_path).split('_')[0].split(' ')[0].split('(')[0].upper()
        
        # Esecuzione della routine di calcolo cinematico
        res = elabora_dati(f_path)        
        
        # Matching tra l'ID elaborato e il corrispettivo record nel master clinico
        clin = master[master['ID'] == sid]
        
        if not clin.empty:
            # Popolamento della struttura dati finale con i valori anagrafici e le feature cinematiche estratte
            pazienti_list.append({   
                'ID': sid,
                'Stato': gruppo_cartella,
                'Sesso': clin['Sesso'].values[0],
                'Eta': clin['Eta'].values[0],
                'Peso': clin['Peso'].values[0],
                'Altezza (cm)': clin['Altezza'].values[0],
                'Durata_SP (s)': round(res['durata'], 2),
                'Velocita_SP (m/s)': round(res['vel'], 3),
                'Falcata_SP (m)': round(res['fal'], 3),
                'Cadenza_SP (passi/min)': round(res['cad'], 2)
            })
    
# Esportazione del dataset consolidato in formato tabulare
df_finale = pd.DataFrame(pazienti_list)
df_finale.to_csv("Tabella_Pazienti_Clinica_Pulita_SP_caminnata.csv", index=False)

In [4]:
#programma da runnare una volta per test (3 totale) modificando le condizioni

PATH_PD_CSV = r"PD PARTICIPANTS\CSV files"  
PATH_HC_CSV = r"CONTROL PARTICIPANTS\CSV files" 
FILE_MASTER_PULITO = r"master_clinico_pulito.csv"
SENSOR_SIZE_M = 0.0127 
SAMPLING_RATE = 0.01 

def elabora_dati(file_path):

        df = pd.read_csv(file_path, sep=None, engine='python')
        df.columns = df.columns.str.strip()
        df_walk = df[df["GeneralEvent"].astype(str).str.contains('Walk')].copy()
        tempo = len(df_walk) * SAMPLING_RATE
        df_walk['L_strike'] = (df_walk["L Foot Contact"].diff(1) == 1)
        df_walk['R_strike'] = (df_walk["R Foot Contact"].diff(1) == 1)
        df_passi = df_walk[df_walk['L_strike'] | df_walk['R_strike']].copy()    
        def get_centroid(val):
            lista_pezzi= str(val).split('|')
            nums = []
            for x in lista_pezzi:
                numero =float(x)
                nums.append(numero)
            return np.mean(nums)*SENSOR_SIZE_M 

        cx = df_passi["Walkway_X"].apply(get_centroid).values
        cy = df_passi["Walkway_Y"].apply(get_centroid).values
        distanza = np.sum(np.sqrt(np.diff(cx)**2 + np.diff(cy)**2))
        return {
            'vel': distanza / tempo, 
            'fal': distanza / (len(df_passi) / 2), 
            'cad': (len(df_passi) / tempo) * 60, 
            'n': len(df_passi),
            'durata': tempo
        }


master = pd.read_csv(FILE_MASTER_PULITO)  
pazienti_list = []
    
for cartella, gruppo_cartella in [(PATH_PD_CSV, 'PD'), (PATH_HC_CSV, 'CO')]:
    files = glob.glob(os.path.join(cartella, "*_hurriedpace.csv")) 
    for f_path in files:
        sid = os.path.basename(f_path).split('_')[0].split(' ')[0].split('(')[0].upper()
        res = elabora_dati(f_path)        
        clin = master[master['ID'] == sid]
        if not clin.empty:
            pazienti_list.append({   
                'ID': sid,
                'Stato': gruppo_cartella,
                'Sesso': clin['Sesso'].values[0],
                'Eta': clin['Eta'].values[0],
                'Peso': clin['Peso'].values[0],
                'Altezza (cm)': clin['Altezza'].values[0],
                'Durata_HP (s)': round(res['durata'], 2),
                'Velocita_HP (m/s)': round(res['vel'], 3),
                'Falcata_HP (m)': round(res['fal'], 3),
                'Cadenza_HP (passi/min)': round(res['cad'], 2)

            })
    
df_finale = pd.DataFrame(pazienti_list)
df_finale.to_csv("Tabella_Pazienti_Clinica_Pulita_HP_caminnata.csv", index=False)

In [5]:
#programma da runnare una volta per test (3 totale) modificando le condizioni

PATH_PD_CSV = r"PD PARTICIPANTS\CSV files"  
PATH_HC_CSV = r"CONTROL PARTICIPANTS\CSV files" 
FILE_MASTER_PULITO = r"master_clinico_pulito.csv"
SENSOR_SIZE_M = 0.0127 
SAMPLING_RATE = 0.01 

def elabora_dati(file_path):

        df = pd.read_csv(file_path, sep=None, engine='python')
        df.columns = df.columns.str.strip()
        df_walk = df[df["GeneralEvent"].astype(str).str.contains('Walk')].copy()
        tempo = len(df_walk) * SAMPLING_RATE
        df_walk['L_strike'] = (df_walk["L Foot Contact"].diff(1) == 1)
        df_walk['R_strike'] = (df_walk["R Foot Contact"].diff(1) == 1)
        df_passi = df_walk[df_walk['L_strike'] | df_walk['R_strike']].copy()    
        def get_centroid(val):
            lista_pezzi= str(val).split('|')
            nums = []
            for x in lista_pezzi:
                numero =float(x)
                nums.append(numero)
            return np.mean(nums)*SENSOR_SIZE_M 

        cx = df_passi["Walkway_X"].apply(get_centroid).values
        cy = df_passi["Walkway_Y"].apply(get_centroid).values
        distanza = np.sum(np.sqrt(np.diff(cx)**2 + np.diff(cy)**2))
        return {
            'vel': distanza / tempo, 
            'fal': distanza / (len(df_passi) / 2), 
            'cad': (len(df_passi) / tempo) * 60, 
            'n': len(df_passi),
            'durata': tempo
        }


master = pd.read_csv(FILE_MASTER_PULITO)  
pazienti_list = []
    
for cartella, gruppo_cartella in [(PATH_PD_CSV, 'PD'), (PATH_HC_CSV, 'CO')]:
    files = glob.glob(os.path.join(cartella, "*_tug.csv")) 
    for f_path in files:
        sid = os.path.basename(f_path).split('_')[0].split(' ')[0].split('(')[0].upper()
        res = elabora_dati(f_path)        
        clin = master[master['ID'] == sid]
        if not clin.empty:
            pazienti_list.append({   
                'ID': sid,
                'Stato': gruppo_cartella,
                'Sesso': clin['Sesso'].values[0],
                'Eta': clin['Eta'].values[0],
                'Peso': clin['Peso'].values[0],
                'Altezza (cm)': clin['Altezza'].values[0],
                'Durata_TUG (s)': round(res['durata'], 2),
                'Velocita_TUG (m/s)': round(res['vel'], 3),
                'Falcata_TUG (m)': round(res['fal'], 3),
                'Cadenza_TUG (passi/min)': round(res['cad'], 2)

            })
    
df_finale = pd.DataFrame(pazienti_list)
df_finale.to_csv("Tabella_Pazienti_Clinica_Pulita_TUG_caminnata.csv", index=False)

### tabelle ASI ###

In [6]:
# DEFINIZIONE DELLE COSTANTI E DEI PATH

percorso_sani = r"CONTROL PARTICIPANTS\CSV files"  # Gruppo di controllo (Healthy Controls)
percorso_pd   = r"PD PARTICIPANTS\CSV files"       # Gruppo patologico (Parkinson's Disease)

# Inizializzazione della struttura dati principale (Dictionary per accesso rapido in fase di update)
pazienti_dict = {}

# Selezione delle colonne di interesse per ottimizzare l'uso della memoria in fase di lettura
# Vengono estratte solo le misurazioni di accelerazione netta depurata dalla gravità
colonne = ['Time', 'GeneralEvent', 'L_Wrist_FreeAcc_E', 'L_Wrist_FreeAcc_N', 'L_Wrist_FreeAcc_U',
           'R_Wrist_FreeAcc_E', 'R_Wrist_FreeAcc_N', 'R_Wrist_FreeAcc_U']


# MAIN PIPELINE: ESTRAZIONE E CALCOLO DELL'ASIMMETRIA

for percorso, gruppo in [(percorso_sani, 'Sano'), (percorso_pd, 'Parkinson')]:
    # Ricerca di tutti i file relativi al task motorio auto-regolato (self-paced)
    nomi_file = glob.glob(os.path.join(percorso, "*_selfpace.csv"))
    
    for strada_file in nomi_file:
        
        # Estrazione e normalizzazione dell'ID univoco del paziente tramite string parsing
        paziente_id = os.path.basename(strada_file).split('_')[0].split(' ')[0].split('(')[0].upper()
        
        # Inizializzazione del record anagrafico per il soggetto corrente
        if paziente_id not in pazienti_dict:
            pazienti_dict[paziente_id] = {
                'ID_Paziente': paziente_id, 
                'Gruppo': gruppo
            }

        try:
            # Acquisizione del dataset con filtro sulle colonne di interesse spaziale
            df = pd.read_csv(strada_file, sep=',', engine='python', usecols=colonne)
            
            # Calcolo della norma euclidea dell'accelerazione per i polsi sinistro e destro.
            # Questo trasforma le tre componenti vettoriali (Est, Nord, Up) in uno scalare 
            # indipendente dall'orientamento locale del sensore.
            df['L_Mag'] = np.sqrt(df['L_Wrist_FreeAcc_E']**2 + df['L_Wrist_FreeAcc_N']**2 + df['L_Wrist_FreeAcc_U']**2)
            df['R_Mag'] = np.sqrt(df['R_Wrist_FreeAcc_E']**2 + df['R_Wrist_FreeAcc_N']**2 + df['R_Wrist_FreeAcc_U']**2)
 
            # Analisi segmentata per le due macro-fasi del protocollo motorio
            for fase in ['Walk', 'Turn']:
                
                segmento = df[df['GeneralEvent'].astype(str).str.contains(fase)].copy()

                # Condizione di validità: il segmento deve esistere e contenere una soglia minima di sample
                if not segmento.empty and len(segmento) > 20:
                    
                    # Trimming del segnale: esclusione del 10% iniziale e finale della finestra temporale.
                    # Questa operazione agisce come filtro per rimuovere i transienti di accelerazione 
                    # associati all'inizio (start-up) e alla fine (decelerazione) del movimento, 
                    # isolando la fase di regime o "steady-state".
                    margine = int(len(segmento) * 0.1)
                    segmento_puro = segmento.iloc[margine:-margine]
                    
                    # Calcolo della deviazione standard della magnitudo come indice di dispersione.
                    # np.nanstd ignora eventuali valori mancanti (NaN) per evitare errori di computazione.
                    # Una deviazione standard maggiore indica oscillazioni (e.g., pendolamento del braccio o tremore) più ampie.
                    std_L = np.nanstd(segmento_puro['L_Mag'].values)
                    std_R = np.nanstd(segmento_puro['R_Mag'].values)

                    # Condizione per evitare divisioni per zero nel calcolo dell'indice
                    if max(std_L, std_R) > 0:
                        # Calcolo dell'Asymmetry Index (ASI) percentuale.
                        # Normalizza la differenza assoluta tra gli arti rispetto all'arto con maggiore escursione.
                        pazienti_dict[paziente_id][f'ASI_{fase}_SP (%)'] = round(abs(std_L - std_R) / max(std_L, std_R) * 100, 3)
        except Exception as e:
            # Gestione silente delle eccezioni (es. file corrotti o colonne mancanti)
            continue


# EXPORT DEI RISULTATI

# Conversione del dictionary nidificato in un DataFrame pandas e salvataggio in formato tabulare
tabella_finale = pd.DataFrame.from_dict(pazienti_dict, orient='index')
tabella_finale.to_csv("Report_Analisi_Asimmetria_Stadi_SP.csv", index=False)

C:\Users\samue\anaconda3\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\samue\anaconda3\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [7]:
percorso_sani = r"CONTROL PARTICIPANTS\CSV files"
percorso_pd   = r"PD PARTICIPANTS\CSV files"
pazienti_dict = {}
colonne = ['Time', 'GeneralEvent', 'L_Wrist_FreeAcc_E', 'L_Wrist_FreeAcc_N', 'L_Wrist_FreeAcc_U',
                 'R_Wrist_FreeAcc_E', 'R_Wrist_FreeAcc_N', 'R_Wrist_FreeAcc_U']
for percorso, gruppo in [(percorso_sani, 'Sano'), (percorso_pd, 'Parkinson')]:
    nomi_file = glob.glob(os.path.join(percorso, "*_hurriedpace.csv"))
    
    for strada_file in nomi_file:
        
        paziente_id = os.path.basename(strada_file).split('_')[0].split(' ')[0].split('(')[0].upper()
        if paziente_id not in pazienti_dict:

            pazienti_dict[paziente_id] = {

                'ID_Paziente': paziente_id, 
                'Gruppo': gruppo
            }

        try:
            
            df = pd.read_csv(strada_file, sep=',', engine='python', usecols=colonne)
            df['L_Mag'] = np.sqrt(df['L_Wrist_FreeAcc_E']**2 + df['L_Wrist_FreeAcc_N']**2 + df['L_Wrist_FreeAcc_U']**2)
            df['R_Mag'] = np.sqrt(df['R_Wrist_FreeAcc_E']**2 + df['R_Wrist_FreeAcc_N']**2 + df['R_Wrist_FreeAcc_U']**2)
 
            for fase in ['Walk', 'Turn']:

                segmento = df[df['GeneralEvent'].str.contains(fase)].copy()

                if not segmento.empty and len(segmento) > 20:
                    margine=int(len(segmento)*0.1)
                    segmento_puro=segmento.iloc[margine:-margine]
                    std_L = np.nanstd(segmento_puro['L_Mag'].values)
                    std_R = np.nanstd(segmento_puro['R_Mag'].values)

                    if max(std_L, std_R) > 0:
                        # Calcolo ASI
                        pazienti_dict[paziente_id][f'ASI_{fase}_HP (%)'] = round(abs(std_L - std_R) / max(std_L, std_R) * 100, 3)
        except:
            continue

tabella_finale = pd.DataFrame.from_dict(pazienti_dict, orient='index')
tabella_finale.to_csv("Report_Analisi_Asimmetria_Stadi_HP.csv", index=False)

In [8]:
percorso_sani = r"CONTROL PARTICIPANTS\CSV files"
percorso_pd   = r"PD PARTICIPANTS\CSV files"
pazienti_dict = {}
colonne = ['Time', 'GeneralEvent', 'L_Wrist_FreeAcc_E', 'L_Wrist_FreeAcc_N', 'L_Wrist_FreeAcc_U',
                 'R_Wrist_FreeAcc_E', 'R_Wrist_FreeAcc_N', 'R_Wrist_FreeAcc_U']
for percorso, gruppo in [(percorso_sani, 'Sano'), (percorso_pd, 'Parkinson')]:
    nomi_file = glob.glob(os.path.join(percorso, "*_tug.csv"))
    
    for strada_file in nomi_file:
        
        paziente_id = os.path.basename(strada_file).split('_')[0].split(' ')[0].split('(')[0].upper()
        if paziente_id not in pazienti_dict:

            pazienti_dict[paziente_id] = {

                'ID_Paziente': paziente_id, 
                'Gruppo': gruppo
            }

        try:
            
            df = pd.read_csv(strada_file, sep=',', engine='python', usecols=colonne)
            df['L_Mag'] = np.sqrt(df['L_Wrist_FreeAcc_E']**2 + df['L_Wrist_FreeAcc_N']**2 + df['L_Wrist_FreeAcc_U']**2)
            df['R_Mag'] = np.sqrt(df['R_Wrist_FreeAcc_E']**2 + df['R_Wrist_FreeAcc_N']**2 + df['R_Wrist_FreeAcc_U']**2)
 
            for fase in ['Walk', 'Turn']:

                segmento = df[df['GeneralEvent'].str.contains(fase)].copy()

                if not segmento.empty and len(segmento) > 20:
                    margine=int(len(segmento)*0.1)
                    segmento_puro=segmento.iloc[margine:-margine]
                    std_L = np.nanstd(segmento_puro['L_Mag'].values)
                    std_R = np.nanstd(segmento_puro['R_Mag'].values)

                    if max(std_L, std_R) > 0:
                        # Calcolo ASI
                        pazienti_dict[paziente_id][f'ASI_{fase}_TUG (%)'] = round(abs(std_L - std_R) / max(std_L, std_R) * 100, 3)
        except:
            continue

tabella_finale = pd.DataFrame.from_dict(pazienti_dict, orient='index')
tabella_finale.to_csv("Report_Analisi_Asimmetria_Stadi_TUG.csv", index=False)

### pulizia dati ###

In [9]:
# 1. LISTA DEI TUOI FILE E DEGLI ID DA ELIMINARE
file_da_elaborare = [
    "Report_Analisi_Asimmetria_Stadi_TUG.csv",
    "Report_Analisi_Asimmetria_Stadi_HP.csv",
    "Report_Analisi_Asimmetria_Stadi_SP.csv",
    "Tabella_Pazienti_Clinica_Pulita_TUG_caminnata.csv",
    "Tabella_Pazienti_Clinica_Pulita_HP_caminnata.csv",
    "Tabella_Pazienti_Clinica_Pulita_SP_caminnata.csv",
    "master_clinico_pulito.csv",
]
id_da_rimuovere = ['WPD029', 'NLS056', 'WPD027']

def pulisci_tutto():
    for nome_file in file_da_elaborare:
        if not os.path.exists(nome_file):
            print(f"File non trovato: {nome_file}, salto.")
            continue
            
        df = pd.read_csv(nome_file)
        colonna_id = 'ID_Paziente' if 'ID_Paziente' in df.columns else 'ID'
        
        def pulisci(val):
            return str(val).split('(')[0].strip().upper()
        
        prima = len(df)
        df_pulito = df[~df[colonna_id].apply(pulisci).isin(id_da_rimuovere)].copy()
        dopo = len(df_pulito)
        
        if prima != dopo:
            df_pulito.to_csv(nome_file, index=False)
            print(f"[OK] {nome_file}: rimosse {prima - dopo} righe.")
        else:
            print(f"[OK] {nome_file}: nessuna riga rimossa.")

pulisci_tutto()
print("\nOperazione completata: tutti i file sono uniformati.")

[OK] Report_Analisi_Asimmetria_Stadi_TUG.csv: rimosse 2 righe.
[OK] Report_Analisi_Asimmetria_Stadi_HP.csv: rimosse 3 righe.
[OK] Report_Analisi_Asimmetria_Stadi_SP.csv: rimosse 3 righe.
[OK] Tabella_Pazienti_Clinica_Pulita_TUG_caminnata.csv: rimosse 2 righe.
[OK] Tabella_Pazienti_Clinica_Pulita_HP_caminnata.csv: rimosse 3 righe.
[OK] Tabella_Pazienti_Clinica_Pulita_SP_caminnata.csv: rimosse 3 righe.
[OK] master_clinico_pulito.csv: rimosse 3 righe.

Operazione completata: tutti i file sono uniformati.


### tabella generale ###

In [10]:
file_list = [
    "Report_Analisi_Asimmetria_Stadi_TUG.csv",
    "Report_Analisi_Asimmetria_Stadi_HP.csv",
    "Report_Analisi_Asimmetria_Stadi_SP.csv",
    "Tabella_Pazienti_Clinica_Pulita_TUG_caminnata.csv",
    "Tabella_Pazienti_Clinica_Pulita_HP_caminnata.csv",
    "Tabella_Pazienti_Clinica_Pulita_SP_caminnata.csv",
    "master_clinico_pulito.csv",
]

cols_cliniche = ['ID', 'Gruppo', 'H&Y', 'Sesso', 'Eta', 'Peso', 'Altezza']

dfs = []
for f in file_list:
    df = pd.read_csv(f)
    df.columns = [c.strip() for c in df.columns]
    df.rename(columns={'ID': 'ID_Paziente', 'Stato': 'Gruppo'}, inplace=True)
    dfs.append(df)

# 3. CREAZIONE MASTER "UNIVERSALE"
all_ids = pd.concat([df[['ID_Paziente']] for df in dfs]).drop_duplicates().dropna()
master_df = all_ids

# 4. RIEMPIMENTO INTELLIGENTE

for df in dfs:
    for col in df.columns:
        if col != 'ID_Paziente':
            if col not in master_df.columns:
                master_df = pd.merge(master_df, df[['ID_Paziente', col]], on='ID_Paziente', how='left')
            else:
                master_df[col] = master_df[col].fillna(df.set_index('ID_Paziente')[col])



# 5. SALVATAGGIO
master_df.to_csv("Dataset_Clinico_Pulito.csv", index=False)
print("Dataset creato con successo! Tutte le cliniche e tutti i dati sono uniti.")

Dataset creato con successo! Tutte le cliniche e tutti i dati sono uniti.
